In [ ]:
from scipy.stats import entropy
from scipy.special import rel_entr
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
import random
import numpy as np
import torch
import gdown

In [ ]:
import pickle
from sklearn.metrics import roc_auc_score

# Logit URLs and Paths.

This is the only cell to be changed when swapping between different datasets

## CIFAR-10 (ULP Models)

In [ ]:
base_url = 'https://drive.google.com/uc?id='


clean_train_min = base_url + '1_nRyX_mK69u95iuqcBWh-V4tEZ1syA-V'
poison_train_min = base_url + '1EACBnWhqB_4qhhJt1yXlXyZGlBiDaRzn'
clean_train_max = base_url + '1jySfnLTdTg3oidiHbd2OzzJmFOJCDQ8j'
poison_train_max = base_url + '1hx0KqcGQIYDJ8tYcgJjq35wCNAAa3VhJ'


clean_test_min = base_url + '11o7g69dpSvOLc96_Wg1G9003M9NtyqGs'
poison_test_min = base_url + '1U5vljmjPgCZ_b7cEjhgV97on-xo8AaMd'
clean_test_max = base_url + '1_sh546aXQrTtG0luS2Z1H_JMrIEqKJpV'
poison_test_max = base_url + '1cZCKl_Z5i9Iv67qPS7jP-GdwzuQ3nDTN'



gdown.download(clean_train_min)
gdown.download(poison_train_min)
gdown.download(clean_train_max)
gdown.download(poison_train_max)
gdown.download(clean_test_min)
gdown.download(poison_test_min)
gdown.download(clean_test_max)
gdown.download(poison_test_max)



train_clean_min_path = '/content/cifar_clean_min_train_logits.pkl'
train_poison_min_path = '/content/cifar_poisoned_min_train_logits.pkl'
train_clean_max_path = '/content/cifar_clean_max_train_logits.pkl'
train_poison_max_path = '/content/cifar_poisoned_max_train_logits.pkl'

test_clean_min_path = '/content/cifar_clean_min_test_logits.pkl'
test_poison_min_path = '/content/cifar_poisoned_min_test_logits.pkl'
test_clean_max_path = '/content/cifar_clean_max_test_logits.pkl'
test_poison_max_path = '/content/cifar_poisoned_max_test_logits.pkl'

Downloading...
From: https://drive.google.com/uc?id=1_nRyX_mK69u95iuqcBWh-V4tEZ1syA-V
To: /content/cifar_clean_min_train_logits.pkl
100%|██████████| 342k/342k [00:00<00:00, 65.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EACBnWhqB_4qhhJt1yXlXyZGlBiDaRzn
To: /content/cifar_poisoned_min_train_logits.pkl
100%|██████████| 342k/342k [00:00<00:00, 67.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jySfnLTdTg3oidiHbd2OzzJmFOJCDQ8j
To: /content/cifar_clean_max_train_logits.pkl
100%|██████████| 342k/342k [00:00<00:00, 96.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hx0KqcGQIYDJ8tYcgJjq35wCNAAa3VhJ
To: /content/cifar_poisoned_max_train_logits.pkl
100%|██████████| 342k/342k [00:00<00:00, 85.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=11o7g69dpSvOLc96_Wg1G9003M9NtyqGs
To: /content/cifar_clean_min_test_logits.pkl
100%|██████████| 68.5k/68.5k [00:00<00:00, 60.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U5vljmjPgCZ_b7cEjhgV97on-

## Our poisoned models (CIFAR-10) (From Exp 3 and 4)

In [ ]:
cifar_resnet_max_clean_test = '1cnCgBDW7BRKP6ju8jhkn2mmkpNSHsQAS'
cifar_resnet_min_clean_test = '1qNfjk2ulwuahiJ4Ve2CBLmRuQucjdaGu'
cifar_resnet_max_poisoned_test = '10SOmtmT_bmR9P7BBKWUEzWoCQbhKgVMm'
cifar_resnet_min_poisoned_test = '1R4XeMvHbFq_azogy5U43MVs__9gF5SHc'

cifar_dense_max_clean_test = '1qtNdJQB-azYxzGFrxZoFsT2Lj0VbwyH8'
cifar_dense_min_clean_test = '1Zwl3izzZGeOOT_ZuZ7ZKo55HoKrdDUEr'
cifar_dense_max_poisoned_test = '1NTBEdNGFfTIMNaisXnEzeD8jjyl7eORF'
cifar_dense_min_poisoned_test = '1nM8cMgBmICeqt5Qm2lEaEx2DfrxmoRa6'

cifar_vit_max_clean_test = '1DnCUHe509Qot-MDvOqtW-x6-ikRkM26k'
cifar_vit_min_clean_test = '1shJSDAxzm2eu7gQgB6OK5wK9QEQKFGTP'
cifar_vit_max_poisoned_test = '1HX4oZWkaycn5Ekk4bwojfYrEFLyy0q5V'
cifar_vit_min_poisoned_test = '1_bcJ94Wz87EPhaHWrVJqVEU1N96ILtOc'

cifar_wanet_min = '1xR7l_2X1gv5PADvljwhNItITq6PVArPZ'
cifar_wanet_max = '1uW-nIAc4Ps07ILqfpZsZvmzQ23W0RB1m'

cifar_sig_min = '1v98XtTYgbQcbBgvyCuvuRmeQwsq5xM3L'
cifar_sig_max = '1oftoIRvGMyLyD6p3PD_qLx2NpN3k8Ct-'


cifar_resnet_large_wanet_min = '1LcDsdzDWwTznKtA-X1iVANjpMR49brF1'
cifar_resnet_large_wanet_max = '1Gbtn3bXxE4w_3Y6d948_sQgGbA8vv93K'

cifar_vgg_mixed_badnet_max = '1HMS5coqPaL0lYwOvNVUQR7t97EAsysYX'
cifar_vgg_mixed_badnet_min = '1iK3veXasxMt5hx3QaDHQunGpVIreciuI'


cifar_resnet_large_clean_min = '1QMCCs-9n2icNvY6oRfUjaVd0oqghyymH'
cifar_resnet_large_clean_max = '1AzqYY_mU90Cyd6KUEyRnquEUP6FkAoWS'


gdown.download(base_url+cifar_resnet_max_clean_test)
gdown.download(base_url+cifar_resnet_min_clean_test)
gdown.download(base_url+cifar_resnet_max_poisoned_test)
gdown.download(base_url+cifar_resnet_min_poisoned_test)
gdown.download(base_url+cifar_dense_max_clean_test)
gdown.download(base_url+cifar_dense_min_clean_test)
gdown.download(base_url+cifar_dense_max_poisoned_test)
gdown.download(base_url+cifar_dense_min_poisoned_test)
gdown.download(base_url+cifar_vit_max_clean_test)
gdown.download(base_url+cifar_vit_min_clean_test)
gdown.download(base_url+cifar_vit_max_poisoned_test)
gdown.download(base_url+cifar_vit_min_poisoned_test)
gdown.download(base_url+cifar_wanet_min)
gdown.download(base_url+cifar_wanet_max)
gdown.download(base_url+cifar_sig_min)
gdown.download(base_url+cifar_sig_max)
gdown.download(base_url+cifar_resnet_large_wanet_min)
gdown.download(base_url+cifar_resnet_large_wanet_max)
gdown.download(base_url+cifar_vgg_mixed_badnet_max)
gdown.download(base_url+cifar_vgg_mixed_badnet_min)
gdown.download(base_url+cifar_resnet_large_clean_min)
gdown.download(base_url+cifar_resnet_large_clean_max)

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1QMCCs-9n2icNvY6oRfUjaVd0oqghyymH

but Gdown can't. Please check connections and permissions.

In [ ]:
# Dict[model_name]: tuple (min_path, max_path, True if poisoned False if clean)

test_model_paths = {
    'SIG':('/content/cifar_sig_min_test_logits.pkl','/content/cifar_sig_max_test_logits.pkl', True),
    'WANET':('/content/cifar_wanet_new_min_test_logits.pkl','/content/cifar_wanet_new_max_test_logits.pkl', True),
    'ResNet Clean':('/content/cifar__clean_resnet_min_test_logits.pkl','/content/cifar__clean_resnet_max_test_logits.pkl', False),
    'ResNet Poisoned':('/content/cifar__poisoned_resnet_min_test_logits.pkl','/content/cifar__poisoned_resnet_max_test_logits.pkl', True),
    'ULP Clean': (test_clean_min_path, test_clean_max_path, False),
    'ULP Poisoned': (test_poison_min_path,test_poison_max_path, True),
    'Dense Clean':('/content/cifar_dense_clean_min_test_logits.pkl','/content/cifar_dense_clean_max_test_logits.pkl', False),
    'Dense Poisoned':('/content/cifar_dense_poisoned_min_test_logits.pkl','/content/cifar_dense_poisoned_max_test_logits.pkl', True),
    'ViT Clean':('/content/cifar_vit_clean_min_test_logits.pkl','/content/cifar_vit_clean_max_test_logits.pkl', False),
    'ViT Poisoned':('/content/cifar_vit_poisoned_min_test_logits.pkl','/content/cifar_vit_poisoned_max_test_logits.pkl', True),
    'ResNet Large WANET':('/content/cifar_resnet_wanet_poisoned_min_test_logits.pkl','/content/cifar_resnet_wanet_poisoned_max_test_logits.pkl', True),
    'Mixed Badnet':('/content/cifar_badnet_mixed_min_test_logits.pkl','/content/cifar_badnet_mixed_max_test_logits.pkl', True),
    'ResNet Large Clean':('/content/cifar_resnet_large_clean_min_test_logits.pkl','/content/cifar_resnet_large_clean_max_test_logits.pkl', False)
}





# Utils


In [ ]:
# Function to load and convert tensors to float16
def load_and_convert(filepath):
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    return {k: v.to(torch.float16) for k, v in data.items()}  # Convert tensors to float16



class Evaluator:
  @staticmethod
  def get_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

  @staticmethod
  def get_confusion_matrix(y_true, y_pred):
    return confusion_matrix(y_true, y_pred)

  @staticmethod
  def get_auc(y_true, y_anomaly_scores):
    return roc_auc_score(y_true, y_anomaly_scores)

  @staticmethod
  def get_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred)

# Detectors

In [ ]:
class GaussianDetector:
    """
    A detector that computes anomaly scores based on Gaussian log probability.

    Configuration parameters:
      - method: 'voting' or 'diagonal'
            (voting: uses softmax over the entire logit matrix,
             diagonal: uses only the main-diagonal entries)
      - standardized: bool, whether to apply standardization to the logits.

    The detector first estimates the mean and std vectors (the "clean signature") from a set of clean models.
    It can then compute an anomaly score for each new model and determine an optimal threshold by comparing
    known clean and poisoned models.
    """
    def __init__(self, method='voting', standardized=False):
        assert method in ['voting', 'diagonal'], f"Invalid method: {method}"
        self.method = method
        self.standardized = standardized

        # Will be set via set_parameters()
        self.clean_signature_means = None
        self.clean_signature_stds = None
        self.optimal_threshold = None

    def compute_feature(self, logits):
        """
        Computes the feature vector for a given logit matrix.

        For method 'voting':
          - If standardized: each row is standardized before applying softmax,
            then the probabilities are averaged over rows.
          - Else: directly apply softmax and take the mean over rows.

        For method 'diagonal':
          - If standardized: standardize the diagonal entries.
          - Else: simply extract the diagonal as the feature.
        """
        eps = 1e-7
        if self.method == 'voting':
            if self.standardized:
                # Standardize each row before softmax.
                logits_std = (logits - logits.mean(dim=1, keepdim=True)) / (logits.std(dim=1, keepdim=True) + eps)
                feature = torch.softmax(logits_std, dim=1).mean(dim=0)
            else:
                feature = torch.softmax(logits, dim=1).mean(dim=0)
        elif self.method == 'diagonal':
            # Extract the diagonal entries (convert to float32 for numerical stability)
            raw_diag = torch.diagonal(logits, dim1=0, dim2=1).to(torch.float32)
            if self.standardized:
                feature = (raw_diag - raw_diag.mean()) / (raw_diag.std() + eps)
            else:
                feature = raw_diag
        else:
            raise ValueError("Invalid method selected.")
        return feature

    def set_parameters(self, clean_logits_tensor):
        """
        Estimates the clean signature parameters from a tensor of clean logits.

        clean_logits_tensor: tensor of shape (num_models, num_classes, num_classes)
        """
        features = []
        for i in range(clean_logits_tensor.size(0)):
            feat = self.compute_feature(clean_logits_tensor[i])
            features.append(feat)
        features = torch.stack(features)  # (num_models, num_classes)
        self.clean_signature_means = features.mean(dim=0)
        self.clean_signature_stds = features.std(dim=0)

    def compute_log_prob(self, feature_vector):
        """
        Computes the log probability of a feature vector under the global Gaussian signature.
        """
        eps = 1e-7
        stds = self.clean_signature_stds.to(torch.float32) + eps
        means = self.clean_signature_means.to(torch.float32)
        # Gaussian log probability (up to a constant)
        log_probs = -0.5 * torch.log(2 * torch.pi * (stds**2)) - ((feature_vector - means)**2 / (2 * stds**2))
        return log_probs.sum()

    def compute_anomaly(self, logits):
        """
        Computes the anomaly score (negative log probability) for given logits.
        """
        feature = self.compute_feature(logits)
        return -self.compute_log_prob(feature)



    def get_optimal_threshold(self, model_instances, percentile=95.0):
        """
        Compute a threshold based on a given percentile of the anomaly scores for normal objects.
        Parameters:
            model_instances: list of InputModel objects.
            percentile: float, the percentile value to use on the normal scores (default 95.0).
        Returns:
            (accuracy, threshold): The accuracy computed on the entire set using this threshold and the threshold value itself.
        Note: In this context, an instance is considered 'poison' (anomaly) if its anomaly score exceeds the threshold.
              The threshold is chosen based solely on the normal (non-poison) objects.
        """
        # Extract anomaly scores and labels (1 if 'poison', 0 if 'normal')
        scores = torch.tensor([model.anomaly_score for model in model_instances])
        labels = torch.tensor([1 if model.model_type == 'poison' else 0 for model in model_instances])

        # Select scores of normal instances (label == 0)
        normal_scores = scores[labels == 0]

        # Calculate the threshold as the given percentile of the normal scores distribution.
        threshold = np.percentile(normal_scores.numpy(), percentile)

        # Classify as 'poison' if the score exceeds the threshold.
        preds = (scores > threshold).int()

        # Compute accuracy over the entire set.
        accuracy = (preds == labels).float().mean().item()

        # Save and return the computed threshold and accuracy.
        self.optimal_threshold = threshold
        return accuracy, threshold



class InputModel:
    """
    A lightweight model wrapper that computes its anomaly score using a given detector.

    Parameters:
       - model_id: an identifier for the model
       - logits: tensor of shape (num_classes, num_classes)
       - model_type: string, either 'clean' or 'poison'
       - detector: an instance of GaussianDetector, used to compute the anomaly score.
    """
    def __init__(self, model_id, logits, model_type, detector: GaussianDetector):
        assert model_type in ['clean', 'poison'], f'Invalid model type: {model_type}'
        self.model_id = model_id
        self.model_type = model_type
        self.anomaly_score = detector.compute_anomaly(logits)



def train_detector(detector, train_clean_path, num_clean_models, percentile, random_seed=42):
    """
    Trains the detector using a subset of clean logits from training data.
    The detector's threshold is estimated from a given percentile of anomaly scores computed
    from the randomly sampled clean models.

    Parameters:
        detector: an instance of GaussianDetector.
        train_clean_path: filepath for clean training data.
        num_clean_models: integer, number of clean models to randomly sample from the loaded data.
        percentile: float, percentile to use for setting the anomaly threshold (e.g., 95.0).
        random_seed: 42 by default.

    Returns:
        anomaly_scores_dict: dictionary containing the anomaly scores for the clean models.
    """
    # Load clean logits.
    clean_logits = load_and_convert(train_clean_path)

    # Set seed for reproducibility.
    np.random.seed(random_seed)

    # Randomly sample keys from the clean logits.
    keys = list(clean_logits.keys())
    sampled_keys = np.random.choice(keys, size=num_clean_models, replace=False)

    # Use the sampled clean logits to set global detector parameters.
    selected_clean_tensors = [clean_logits[key] for key in sampled_keys]
    clean_tensor = torch.stack(selected_clean_tensors)
    detector.set_parameters(clean_tensor)

    # Construct training models using only the sampled clean models.
    training_models = []
    for key in sampled_keys:
        training_models.append(InputModel(key, clean_logits[key], 'clean', detector))

    # Optimize threshold using the specified percentile.
    train_acc, optimal_threshold = detector.get_optimal_threshold(training_models, percentile)

    # Evaluate performance on the sampled clean models.
    y_anomaly_scores = []
    anomaly_scores_dict = {'clean': []}

    for m in training_models:
        y_anomaly_scores.append(m.anomaly_score)
        anomaly_scores_dict['clean'].append(m.anomaly_score)


    anomaly_scores_dict['clean'] = np.array(anomaly_scores_dict['clean'])
    return anomaly_scores_dict


def test_detector(detector, test_logits_path, poison = True):
    """
    Evaluates the detector on test data.

    detector: an instance of GaussianDetector with parameters and threshold previously set.
    test_logits_path: filepaths for test data.

    Returns a dictionary of anomaly scores separated by model type.
    """
    # Load test data.
    test_logits = load_and_convert(test_logits_path)
    test_logits = dict(list(test_logits.items())[-100:]) # get last 100 only


    test_models = []
    for model_id in test_logits.keys():
        if poison:
            test_models.append(InputModel(model_id, test_logits[model_id], 'poison', detector))
        else:
            test_models.append(InputModel(model_id, test_logits[model_id], 'clean', detector))


    y_true = []
    y_pred = []
    y_anomaly_scores = []
    model_counts_dict = {'clean': 0, 'poison': 0}

    for m in test_models:
        y_true.append(1 if m.model_type == 'poison' else 0)
        y_pred.append(1 if m.anomaly_score > detector.optimal_threshold else 0)
        y_anomaly_scores.append(m.anomaly_score)
        if m.anomaly_score > detector.optimal_threshold:
            model_counts_dict['poison'] += 1
        else:
            model_counts_dict['clean'] += 1


    return y_anomaly_scores, model_counts_dict



# MinMax Ensemble

In [ ]:
class MinMaxEnsembler:
  optimal_threshold = None

  def __init__(self, method='sum scores', standardized=False):
        assert method in ['sum scores'], f"Invalid method: {method}"
        self.method = method


  def get_optimal_threshold(self, min_anomaly_scores_dict, max_anomaly_scores_dict, percentile=95.0):
      """
      Compute a threshold based on a given percentile of the anomaly scores for normal objects.
      Parameters:
          min_anomaly_scores_dict: {'clean': (num_models,), 'poison': (num_models,)}
          max_anomaly_scores_dict: {'clean': (num_models,), 'poison': (num_models,)}
          percentile: float, the percentile value to use on the normal scores (default 95.0).
      Returns:
          (accuracy, threshold): The accuracy computed on the entire set using this threshold and the threshold value itself.
      Note: In this context, an instance is considered 'poison' (anomaly) if its anomaly score exceeds the threshold.
            The threshold is chosen based solely on the normal (non-poison) objects.
      """
      # Extract anomaly scores and labels (1 if 'poison', 0 if 'normal')
      if self.method == 'sum scores':
        clean_summed_scores = min_anomaly_scores_dict['clean'] + max_anomaly_scores_dict['clean']

      # Calculate the threshold as the given percentile of the normal scores distribution.
      threshold = np.percentile(clean_summed_scores, percentile)

      # Save and return the computed threshold and accuracy.
      self.optimal_threshold = threshold
      return threshold


def train_both(ensembler, train_min_anomaly_scores, train_max_anomaly_scores):
  ensembler.get_optimal_threshold(train_min_anomaly_scores, train_max_anomaly_scores)
  num_models = len(train_min_anomaly_scores['clean'])
  y_anomaly_scores = []

  for model_type in train_min_anomaly_scores.keys():
    for model_idx in range(num_models):
      y_anomaly_scores.append(train_min_anomaly_scores[model_type][model_idx] + train_max_anomaly_scores[model_type][model_idx])

  return y_anomaly_scores




def test_both(ensembler, test_min_anomaly_scores, test_max_anomaly_scores):
  num_models = len(test_min_anomaly_scores)
  y_true = []
  y_pred = []
  y_anomaly_scores = []
  model_counts_dict = {'clean': 0, 'poison': 0}

  for model_idx in range(num_models):
    y_anomaly_scores.append(test_min_anomaly_scores[model_idx] + test_max_anomaly_scores[model_idx])
    if test_min_anomaly_scores[model_idx] + test_max_anomaly_scores[model_idx] > ensembler.optimal_threshold:
      model_counts_dict['poison'] += 1
    else:
      model_counts_dict['clean'] += 1

  return y_anomaly_scores, model_counts_dict


# Experiment

## Experiment Array

In [ ]:
import random
from tqdm.auto import tqdm

seed = 42
print("Seeds: ", seed)

num_clean_models = 500
print("num_clean_models: ", num_clean_models)

percentile = 95
print("percentiles: ", percentile)

# Train
standardization = True
print("standardization: ", standardization)
print("Last 100")
min_detector = GaussianDetector(method='voting', standardized=standardization)
max_detector = GaussianDetector(method='diagonal', standardized=standardization)
min_max_ensemble = MinMaxEnsembler()
min_train_scores = train_detector(min_detector, train_clean_min_path, num_clean_models = num_clean_models, percentile=percentile, random_seed=seed)
max_train_scores = train_detector(max_detector, train_clean_max_path, num_clean_models = num_clean_models, percentile=percentile, random_seed=seed)
train_both(min_max_ensemble, min_train_scores, max_train_scores)


# Test
for model_name, model_paths in test_model_paths.items():
  min_paths, max_paths, is_poisoned = model_paths
  print(model_paths)
  min_anomaly_scores, min_models_dict = test_detector(min_detector, min_paths, False)
  max_anomaly_scores, max_models_dict = test_detector(max_detector, max_paths, False)
  both_anomaly_scores, both_counts_dict, = test_both(min_max_ensemble, min_anomaly_scores, max_anomaly_scores)
  print(model_name)
  print('Min')
  print(min_models_dict)
  print('Max')
  print(max_models_dict)
  print('Both')
  print(both_counts_dict)
  print("\n\n")






Seeds:  42
num_clean_models:  500
percentiles:  95
standardization:  True
Last 100
('/content/cifar_sig_min_test_logits.pkl', '/content/cifar_sig_max_test_logits.pkl', True)
SIG
Min
{'clean': 38, 'poison': 62}
Max
{'clean': 60, 'poison': 40}
Both
{'clean': 37, 'poison': 63}



('/content/cifar_wanet_new_min_test_logits.pkl', '/content/cifar_wanet_new_max_test_logits.pkl', True)
WANET
Min
{'clean': 84, 'poison': 16}
Max
{'clean': 7, 'poison': 93}
Both
{'clean': 15, 'poison': 85}



('/content/cifar__clean_resnet_min_test_logits.pkl', '/content/cifar__clean_resnet_max_test_logits.pkl', False)
ResNet Clean
Min
{'clean': 95, 'poison': 5}
Max
{'clean': 74, 'poison': 26}
Both
{'clean': 87, 'poison': 13}



('/content/cifar__poisoned_resnet_min_test_logits.pkl', '/content/cifar__poisoned_resnet_max_test_logits.pkl', True)
ResNet Poisoned
Min
{'clean': 34, 'poison': 66}
Max
{'clean': 26, 'poison': 74}
Both
{'clean': 16, 'poison': 84}



('/content/cifar_clean_min_test_logits.pkl', '/content/ci